# Datos reales: lo que el fixture no puede probar

El fixture es sintetico, diario y completo: veinte columnas sin un solo hueco y
con las mismas fechas. Por eso hay cuatro funciones del motor que nunca se
ejercitan con el, porque solo tienen trabajo cuando los datos vienen sucios.

| funcion | para que sirve | por que el fixture no la prueba |
|---|---|---|
| `fetch_prices` | descargar de yfinance | el fixture es un CSV local |
| `coverage` | medir huecos y rangos | no hay huecos ni rangos distintos |
| `align` | recortar a fechas comunes | todas las columnas ya coinciden |
| `resample_prices` | pasar a semanal o mensual | el fixture ya viene diario |

Este notebook las prueba contra el mercado de verdad.

> **Se necesita conexion a internet.** Si estas sin red, las celdas de descarga
> fallan y el resto no tiene datos con que trabajar.

In [23]:
import numpy as np
import pandas as pd

import src.contracts as ct
import src.data as dt
import src.analytics as ana
import src.forecasting as fc
import src.risk_rules as rr
import src.compare as cp

params = ct.Params()
INICIO, FIN = "2021-01-01", "2025-01-01"

## 1. La descarga

`fetch_prices` baja los tickers **uno a uno**, no en bloque. Es mas lento, pero a
cambio un simbolo invalido no arrastra a los demas: cada fallo queda aislado en
su propia entrada del diccionario de errores.

Mezclamos a proposito tres cosas distintas:

- tres tickers de EEUU que deberian funcionar
- **SAP.DE**, aleman: cotiza en otro mercado, con **festivos distintos**


In [24]:
TICKERS = ["AAPL", "MSFT", "KO", "SAP.DE", "EIA_INVALID_2026"]

precios, errores = dt.fetch_prices(TICKERS, INICIO, FIN)

print("respondieron:", list(precios.columns))
print("fallaron:", len(errores))
for t, motivo in errores.items():
    print(f"   {t}: {motivo}")

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: EIA_INVALID_2026"}}}
$EIA_INVALID_2026: possibly delisted; no timezone found

1 Failed download:
['EIA_INVALID_2026']: possibly delisted; no timezone found


respondieron: ['AAPL', 'MSFT', 'KO', 'SAP.DE']
fallaron: 1
   EIA_INVALID_2026: sin datos en el rango pedido


Lo que exige `FAIL-20P1-01`: *"Invalid ticker must not delete valid results"*.
El ticker inventado debe aparecer en `errores` y los validos seguir en `precios`,
intactos.

Si no hay fallos, `errores` llega vacio — un diccionario vacio es falso en Python,
asi que `if errores:` basta para saber si hubo problemas.

In [25]:
precios.tail()

,AAPL,MSFT,KO,SAP.DE
Date,,,,
2024-12-24,256.339783,433.363525,60.233418,NaN
2024-12-26,257.153809,432.160126,59.974625,NaN
2024-12-27,253.748535,424.683075,59.859596,233.393478
2024-12-30,250.382950,419.060455,59.457016,230.275070
2024-12-31,248.615768,415.775665,59.677479,NaN


## 2. Cobertura

Aqui `coverage` por fin tiene algo que decir. En el fixture todas las filas eran
identicas; con datos reales cada mercado tiene su calendario.

In [26]:
dt.coverage(precios)

,start,end,n,missing_pct
AAPL,2021-01-04,2024-12-31,1005,2.710552
MSFT,2021-01-04,2024-12-31,1005,2.710552
KO,2021-01-04,2024-12-31,1005,2.710552
SAP.DE,2021-01-04,2024-12-30,1021,1.161665


Fijate en `missing_pct` de **SAP.DE**: los dias que la bolsa alemana abre y la
estadounidense no (y al reves) aparecen como huecos, porque el indice es la union
de todas las fechas vistas.

Ese numero es el filtro natural antes de comparar: un activo con demasiados
faltantes no deberia entrar al mapa media-volatilidad.

## 3. Alinear

`align` deja solo las fechas donde **todos** tienen dato. Es destructivo a
proposito, y por eso solo se usa en la vista comparativa: para analizar un activo
suelto no tiene sentido tirar sus fechas buenas porque a otro le falten.

In [27]:
alineado = dt.align(precios)

print(f"antes  {len(precios)} fechas")
print(f"despues {len(alineado)} fechas")
print(f"se perdieron {len(precios) - len(alineado)} "
      f"({(1 - len(alineado) / len(precios)):.1%})")

antes  1033 fechas
despues 993 fechas
se perdieron 40 (3.9%)


In [28]:
# que fechas se cayeron y a quien le faltaba el dato
perdidas = precios.index.difference(alineado.index)
precios.loc[perdidas].head(10)

,AAPL,MSFT,KO,SAP.DE
Date,,,,
2021-01-18,NaN,NaN,NaN,93.507324
2021-02-15,NaN,NaN,NaN,98.579391
2021-04-05,122.407593,238.032883,45.208416,NaN
2021-05-24,123.784218,240.220474,46.911961,NaN
2021-05-31,NaN,NaN,NaN,104.536331
2021-07-05,NaN,NaN,NaN,109.490570
2021-09-06,NaN,NaN,NaN,117.557335
2021-11-25,NaN,NaN,NaN,110.337769
2021-12-31,173.449448,323.365967,51.853554,NaN


## 4. Remuestreo

`resample_prices` toma el **ultimo precio valido** de cada periodo. La guia lo
marca como error tipico: hay que remuestrear **precios** y despues calcular
rendimientos, nunca calcular rendimientos diarios y luego sumarlos por semana.

In [29]:
serie = alineado["AAPL"]

for freq in ("D", "W-FRI", "ME"):
    s = dt.resample_prices(serie, freq)
    g = ana.log_returns(s)
    m = dt.PERIODOS_POR_ANIO[freq]
    media, sd = ana.annualize(*ana.sample_stats(g), m)
    print(f"{freq:<6} {len(s):>4} precios  {len(g):>4} rendimientos   "
          f"m={m:<4} media anual {media: .4%}   volatilidad {sd:.4%}")

D       993 precios   992 rendimientos   m=252  media anual  17.5188%   volatilidad 26.6988%
W-FRI   209 precios   208 rendimientos   m=52   media anual  16.7359%   volatilidad 25.4990%
ME       48 precios    47 rendimientos   m=12   media anual  17.1094%   volatilidad 25.2778%


Las tres filas describen **el mismo activo en el mismo periodo**. Si la
anualizacion esta bien hecha, los tres numeros deberian parecerse — no ser
identicos, porque cada frecuencia ve una muestra distinta, pero si estar en el
mismo orden de magnitud.

Que la volatilidad anual cambie mucho entre frecuencias es una senal de que los
rendimientos no son independientes: es el mismo efecto que hace que el VaR
parametrico a 20 dias se quede corto.

In [30]:
# el camino correcto contra el prohibido, medido
semanal = ana.log_returns(dt.resample_prices(serie, "W-FRI"))
diario = ana.log_returns(serie)
agregado = diario.resample("W-FRI").sum()

comparacion = pd.DataFrame({"correcto": semanal, "agregando_diarios": agregado})
comparacion["diferencia"] = comparacion.correcto - comparacion.agregando_diarios

print(f"semanas con diferencia no nula: "
      f"{(comparacion.diferencia.abs() > 1e-12).sum()} de {len(comparacion)}")
print(f"diferencia maxima: {comparacion.diferencia.abs().max():.6e}")
comparacion.head()

semanas con diferencia no nula: 0 de 209
diferencia maxima: 3.439089e-16


,correcto,agregando_diarios,diferencia
Date,,,
2021-01-08,NaN,0.020195,NaN
2021-01-15,-0.037892,-0.037892,-4.857226e-17
2021-01-22,0.089688,0.089688,-1.387779e-17
2021-01-29,-0.052479,-0.052479,-4.857226e-17
2021-02-05,0.037222,0.037222,6.938894e-17


## 5. El motor completo, con datos reales

Las mismas llamadas del otro notebook, sobre precios que nadie fabrico. Aqui los
diagnosticos suelen rechazar mas y las advertencias aparecen de verdad.

In [31]:
TICKER, H = "AAPL", 20
M = dt.PERIODOS_POR_ANIO["D"]

s = precios[TICKER].dropna()
g = ana.log_returns(s)
P_t = s.iloc[-1]

diag = ana.diagnostics(g)
mu, sigma = fc.fit(g, "B")
m_H, v_H = fc.cumulative_moments(mu, sigma, H)
niveles = rr.levels(P_t, m_H, v_H, params.p_L, params.p_U,
                    params.cost_buy, params.cost_sell)
wf = fc.walk_forward(g, "B", H, M, params)
puerta = rr.signal_gate(niveles, diag, wf["sufficient"], params.BR_min)

print(f"{TICKER}  n={len(g)}  P_t={P_t:.2f}")
print("rechazan   ", [k for k in ("jarque_bera", "ljung_box", "brown_forsythe",
                                  "arch_lm") if diag[k]["reject"]])
if wf["sufficient"]:
    print("cobertura   ", wf["coverage"], " (prometia 0.90)")
else:
    print(f"walk-forward: insuficiente - {wf['T']} rendimientos, "
          f"hacen falta {wf['required']}")
print("señal     ", puerta["signal"], "|", puerta["label"])
print("incumplidas ", puerta["failed"] or "ninguna")
print("advertencias", puerta["warnings"] or "ninguna")

AAPL  n=1004  P_t=248.62
rechazan    ['jarque_bera', 'brown_forsythe', 'arch_lm']
cobertura    1.0  (prometia 0.90)
señal      True | condicional
incumplidas  ninguna
advertencias ['jarque_bera', 'brown_forsythe', 'arch_lm']


## 6. Comparación

Con cuatro activos reales en vez de veinte sinteticos. La frontera sera mas
pequeña, pero la logica es identica.

In [32]:
comp = cp.preselect(cp.summary(alineado, M))
comp[["mean_log_annual", "volatility_annual", "individual_rvr",
      "non_dominated", "selected_max_rvr"]].sort_values("individual_rvr",
                                                        ascending=False)

,mean_log_annual,volatility_annual,individual_rvr,non_dominated,selected_max_rvr
ticker,,,,,
SAP.DE,0.223876,0.22997,0.973498,True,True
MSFT,0.178475,0.262356,0.680278,False,False
AAPL,0.175188,0.266988,0.656167,False,False
KO,0.071944,0.155274,0.463334,True,False
